# 04 - Fine tuning de ResNet18

Aplicamos transfer learning: empezamos con todo congelado salvo la cabeza,
luego descongelamos las ultimas capas y entrenamos con LR discriminativos
(uno para la cabeza y otro mas pequenio para el backbone descongelado).

Etapas:
1. Setup, config y dataloaders.
2. Construccion del modelo y politica de freezing.
3. Entrenamiento con W&B opcional y validacion por epoch.
4. Evaluacion final en validacion, checkpoint y registro final.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import torch
from torch import nn, optim
from sklearn.metrics import f1_score

from src.models.transfer import build_resnet18_finetune
from src.training.engine import evaluate_classification, train_one_epoch
from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "finetune.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else config["device"])
print("Device:", device)
config

Device: cuda


{'experiment_name': 'finetune_resnet18',
 'seed': 42,
 'device': 'cpu',
 'data': {'root_dir': 'data/asl_alphabet_v1/processed',
  'train_subdir': 'train',
  'val_subdir': 'val',
  'image_size': 224,
  'batch_size': 32,
  'num_workers': 0},
 'training': {'epochs': 10,
  'learning_rate_head': 0.001,
  'learning_rate_backbone': 0.0001,
  'weight_decay': 0.0},
 'model': {'backbone': 'resnet18',
  'freeze_backbone': True,
  'unfreeze_last_n_layers': 1,
  'num_classes': None},
 'tracking': {'use_wandb': True,
  'project': 'signlanguage-classifier',
  'run_name': 'finetune-resnet18',
  'tags': ['finetune', 'transfer-learning']},
 'output': {'artifacts_dir': 'artifacts',
  'checkpoint_name': 'finetune_resnet18.pt'}}

## Data loaders

In [2]:
import torch
from torch.utils.data import DataLoader

from src.data.dataset import load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "num_classes": num_classes})


{'train': 189641, 'val': 33461, 'num_classes': 29}


## Modelo preentrenado y optimizador con LR discriminativos

In [3]:
model = build_resnet18_finetune(
    num_classes=num_classes,
    freeze_backbone=config["model"].get("freeze_backbone", True),
    unfreeze_last_n_layers=config["model"].get("unfreeze_last_n_layers", 1),
).to(device)

head_params = [p for n, p in model.named_parameters() if n.startswith("fc.") and p.requires_grad]
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.") and p.requires_grad]

param_groups = [{"params": head_params, "lr": config["training"]["learning_rate_head"]}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": config["training"]["learning_rate_backbone"]})

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(param_groups, weight_decay=config["training"].get("weight_decay", 0.0))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Trainable params: 8,408,605 / 11,191,389


## Loop de fine tuning

In [4]:
from tqdm import tqdm

run = init_wandb_run(
    config=config,
    enabled=config["tracking"].get("use_wandb", False),
    project=config["tracking"]["project"],
    run_name=config["tracking"].get("run_name", config["experiment_name"]),
    tags=config["tracking"].get("tags"),
)


def train_one_epoch_with_progress(model, loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_targets = []

    train_bar = tqdm(
        loader,
        total=len(loader),
        desc=f"Epoch {epoch:02d} [train]",
        leave=True,
        unit="batch",
        dynamic_ncols=True,
        mininterval=0.2,
    )
    for batch_idx, (images, labels) in enumerate(train_bar, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

        running_loss = total_loss / max(total_samples, 1)
        running_acc = total_correct / max(total_samples, 1)
        train_bar.set_postfix(batch=f"{batch_idx}/{len(loader)}", loss=f"{running_loss:.4f}", acc=f"{running_acc:.4f}")

    train_bar.close()
    train_loss = total_loss / max(total_samples, 1)
    train_acc = total_correct / max(total_samples, 1)
    train_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return train_loss, train_acc, train_f1


@torch.no_grad()
def evaluate_with_progress(model, loader, criterion, device, epoch):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_targets = []

    val_bar = tqdm(
        loader,
        total=len(loader),
        desc=f"Epoch {epoch:02d} [val]",
        leave=True,
        unit="batch",
        dynamic_ncols=True,
        mininterval=0.2,
    )
    for batch_idx, (images, labels) in enumerate(val_bar, start=1):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

        running_loss = total_loss / max(total_samples, 1)
        running_acc = total_correct / max(total_samples, 1)
        val_bar.set_postfix(batch=f"{batch_idx}/{len(loader)}", loss=f"{running_loss:.4f}", acc=f"{running_acc:.4f}")

    val_bar.close()
    val_loss = total_loss / max(total_samples, 1)
    val_acc = total_correct / max(total_samples, 1)
    val_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return val_loss, val_acc, val_f1


history = []
epoch_bar = tqdm(
    range(1, config["training"]["epochs"] + 1),
    desc="Fine-tuning epochs",
    unit="epoch",
    leave=True,
    dynamic_ncols=True,
)
for epoch in epoch_bar:
    train_loss, train_acc, train_f1_macro = train_one_epoch_with_progress(model, train_loader, criterion, optimizer, device, epoch)
    val_loss, val_acc, val_f1_macro = evaluate_with_progress(model, val_loader, criterion, device, epoch)

    entry = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1_macro": train_f1_macro,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1_macro,
    }
    history.append(entry)
    if run is not None:
        run.log(entry)

    epoch_bar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        train_f1=f"{train_f1_macro:.4f}",
        val_loss=f"{val_loss:.4f}",
        val_acc=f"{val_acc:.4f}",
        val_f1=f"{val_f1_macro:.4f}",
    )
    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_f1_macro={train_f1_macro:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1_macro={val_f1_macro:.4f}"
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 202513902 (adne-image-classification) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Fine-tuning epochs:  10%|█         | 1/10 [17:14<2:35:08, 1034.25s/epoch, train_loss=0.0832, val_acc=0.9948, val_loss=0.0177]

Epoch 01 | train_loss=0.0832 val_loss=0.0177 val_acc=0.9948


Fine-tuning epochs:  20%|██        | 2/10 [32:24<2:08:11, 961.45s/epoch, train_loss=0.0158, val_acc=0.9967, val_loss=0.0111] 

Epoch 02 | train_loss=0.0158 val_loss=0.0111 val_acc=0.9967


...

Epoch 08 | train_loss=0.0054 val_loss=0.0047 val_acc=0.9991


Fine-tuning epochs:  90%|█████████ | 9/10 [2:30:27<16:38, 998.49s/epoch, train_loss=0.0048, val_acc=0.9987, val_loss=0.0040] 

Epoch 09 | train_loss=0.0048 val_loss=0.0040 val_acc=0.9987


Fine-tuning epochs: 100%|██████████| 10/10 [2:47:20<00:00, 1004.07s/epoch, train_loss=0.0054, val_acc=0.9986, val_loss=0.0051]

Epoch 10 | train_loss=0.0054 val_loss=0.0051 val_acc=0.9986


## Evaluacion final en validacion y guardado del checkpoint

In [5]:
final_val_loss, final_val_acc = evaluate_classification(model, val_loader, criterion, device)

all_preds = []
all_targets = []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

final_val_f1_macro = f1_score(all_targets, all_preds, average="macro", zero_division=0)
print({
    "final_val_loss": final_val_loss,
    "final_val_accuracy": final_val_acc,
    "final_val_f1_macro": final_val_f1_macro,
})

output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "finetune"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / config["output"]["checkpoint_name"]
torch.save({
    "state_dict": model.state_dict(),
    "model_type": "resnet18_finetune",
    "class_names": class_names,
    "image_size": image_size,
    "backbone": config["model"].get("backbone", "resnet18"),
    "freeze_backbone": config["model"].get("freeze_backbone", True),
    "unfreeze_last_n_layers": config["model"].get("unfreeze_last_n_layers", 1),
}, checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

if run is not None:
    run.log({
        "final_val_loss": final_val_loss,
        "final_val_accuracy": final_val_acc,
        "final_val_f1_macro": final_val_f1_macro,
    })
    finish_wandb_run(run)

{'final_val_loss': 0.005103272131141371, 'final_val_accuracy': 0.9985654941573773}
Saved checkpoint to c:\Users\aleja\Documents\Comillas\2º Cuatri\Análisis de Datos no Estructurados\signlanguage-classifier\artifacts\finetune\finetune_resnet18.pt


epoch,▁▂▃▃▄▅▆▆▇█
final_val_accuracy,▁
final_val_loss,▁
train_accuracy,▁▇▇███████
train_loss,█▂▂▁▁▁▁▁▁▁
val_accuracy,▂▅▅▁▇▆▇█▇▇
val_loss,▇▄▄█▂▂▂▁▁▂
epoch,10
final_val_accuracy,0.99857
final_val_loss,0.0051
train_accuracy,0.9987


## Evaluacion de checkpoint .pt sin reentrenar

Carga un `.pt` desde `artifacts/` y calcula metricas en `train` y `val` sin ejecutar el loop de entrenamiento.

In [8]:
from sklearn.metrics import f1_score
from tqdm import tqdm

from src.inference.predict import load_torch_model


def resolve_checkpoint_path() -> Path:
    # 1) Use configured checkpoint when available.
    configured = PROJECT_ROOT / config["output"]["artifacts_dir"] / "finetune" / config["output"]["checkpoint_name"]
    if configured.exists():
        return configured

    # 2) Fallback to latest .pt/.pth/.ckpt under artifacts.
    artifacts_root = PROJECT_ROOT / config["output"]["artifacts_dir"]
    candidates = [
        p
        for p in artifacts_root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".pt", ".pth", ".ckpt"}
    ]
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {artifacts_root}")

    return max(candidates, key=lambda p: p.stat().st_mtime)


@torch.no_grad()
def evaluate_split_with_f1(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    split_name: str,
) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_targets = []

    bar = tqdm(
        loader,
        total=len(loader),
        desc=f"Evaluating {split_name}",
        unit="batch",
        leave=True,
        dynamic_ncols=True,
        mininterval=0.2,
    )
    for images, labels in bar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

        running_loss = total_loss / max(total_samples, 1)
        running_acc = total_correct / max(total_samples, 1)
        bar.set_postfix(loss=f"{running_loss:.4f}", acc=f"{running_acc:.4f}")

    bar.close()

    loss_value = total_loss / max(total_samples, 1)
    acc_value = total_correct / max(total_samples, 1)
    f1_value = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return {
        "loss": loss_value,
        "accuracy": acc_value,
        "f1_macro": f1_value,
    }


checkpoint_path = resolve_checkpoint_path()
checkpoint_model, ckpt_class_names, ckpt_image_size = load_torch_model(checkpoint_path=checkpoint_path, device=device)
criterion_eval = nn.CrossEntropyLoss()

if list(class_names) != list(ckpt_class_names):
    print("[WARN] class_names de dataset y checkpoint no coinciden exactamente.")
if image_size != ckpt_image_size:
    print(f"[WARN] image_size loader={image_size} y checkpoint={ckpt_image_size}.")

train_metrics = evaluate_split_with_f1(checkpoint_model, train_loader, criterion_eval, device, split_name="train")
val_metrics = evaluate_split_with_f1(checkpoint_model, val_loader, criterion_eval, device, split_name="val")

metrics_report = {
    "checkpoint_path": str(checkpoint_path),
    "train_loss": train_metrics["loss"],
    "train_accuracy": train_metrics["accuracy"],
    "train_f1_macro": train_metrics["f1_macro"],
    "val_loss": val_metrics["loss"],
    "val_accuracy": val_metrics["accuracy"],
    "val_f1_macro": val_metrics["f1_macro"],
}
print(metrics_report)

if "run" in globals() and run is not None:
    try:
        run.log(metrics_report)
    except Exception as error:
        print(f"[INFO] No se pudo loguear en W&B (probablemente run finalizada): {error}")

Evaluating val: 100%|██████████| 1046/1046 [09:21<00:00,  1.86batch/s, acc=0.9986, loss=0.0051]

{'checkpoint_path': 'c:\\Users\\aleja\\Documents\\Comillas\\2º Cuatri\\Análisis de Datos no Estructurados\\signlanguage-classifier\\artifacts\\finetune\\finetune_resnet18.pt', 'train_loss': 0.0010443425190681812, 'train_accuracy': 0.9996836127208779, 'train_f1_macro': 0.9996926960075952, 'val_loss': 0.005103272131141371, 'val_accuracy': 0.9985654941573773, 'val_f1_macro': 0.9985888337622637}
[INFO] No se pudo loguear en W&B (probablemente run finalizada): Run (lwnpjpao) is finished. The call to `log` will be ignored. Please make sure that you are using an active run.
